<a href="https://colab.research.google.com/github/cikobaba/Sentiment-Analysis-BERT-Comparison/blob/main/distilbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning BERT (and friends) for multi-label text classification

In this notebook, we are going to fine-tune BERT to predict one or more labels for a given piece of text. Note that this notebook illustrates how to fine-tune a bert-base-uncased model, but you can also fine-tune a RoBERTa, DeBERTa, DistilBERT, CANINE, ... checkpoint in the same way.

All of those work in the same way: they add a linear layer on top of the base model, which is used to produce a tensor of shape (batch_size, num_labels), indicating the unnormalized scores for a number of labels for every example in the batch.



## Set-up environment

First, we install the libraries which we'll use: HuggingFace Transformers and Datasets. We'll use `uv` for package management. If running on Google Colab, you can use `pip install` instead.

In [ ]:
!uv pip install transformers datasets scikit-learn torch pandas --quiet

In [ ]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.8 MB/s eta 0:00:00


In [ ]:
import os

for f in os.listdir():
    print(f)

.config
shopify_reviews.xlsx
sample_data


In [ ]:
import pandas as pd

df = pd.read_excel("shopify_reviews.xlsx")
df.head()

,content,email,product_id,status,name,photo_urls,created_at,rating,product_title,product_sku,product_url,product_handle
0,i used to have a feliway - they can be interch...,NaN,15735008788853,published,Halina Król,NaN,2025-08-17T00:00:00.000Z,5,Cats Calming Diffuser Kit,NaN,https://us1cxf-vd.myshopify.com/products/cats-...,cats-calming-diffuser-kit
1,NaN,NaN,15735008788853,published,Daniel Castro,NaN,2026-02-15T00:00:00.000Z,5,Cats Calming Diffuser Kit,NaN,https://us1cxf-vd.myshopify.com/products/cats-...,cats-calming-diffuser-kit
2,NaN,NaN,15735008788853,published,Mariana Pérez,NaN,2025-12-05T00:00:00.000Z,5,Cats Calming Diffuser Kit,NaN,https://us1cxf-vd.myshopify.com/products/cats-...,cats-calming-diffuser-kit
3,"Excelente producto, los gatos se han calmado e...",NaN,15735008788853,published,Sofía Silva,NaN,2025-12-29T00:00:00.000Z,5,Cats Calming Diffuser Kit,NaN,https://us1cxf-vd.myshopify.com/products/cats-...,cats-calming-diffuser-kit
4,NaN,NaN,15707298595189,published,Noa Cohen,NaN,2026-02-20T00:00:00.000Z,5,PetPacify Dog Calming Diffuser,NaN,https://us1cxf-vd.myshopify.com/products/petpa...,petpacify-dog-calming-diffuser-kit


In [ ]:
import os

file_name = 'shopify_reviews.xlsx'
if os.path.exists(file_name):
    print(f"The file '{file_name}' exists in the current directory.")
else:
    print(f"The file '{file_name}' does NOT exist in the current directory.")

The file 'shopify_reviews.xlsx' exists in the current directory.


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

df = pd.read_excel("shopify_reviews.xlsx")
df = df[["content", "rating"]].copy()
df.head()

,content,rating
0,i used to have a feliway - they can be interch...,5
1,NaN,5
2,NaN,5
3,"Excelente producto, los gatos se han calmado e...",5
4,NaN,5


In [ ]:
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = " ".join(x.split())
    if len(x) == 0:
        return None
    return x

def map_label(rating):
    try:
        r = float(rating)
    except:
        return None

    if r in [1.0, 2.0]:
        return 0   # negative
    elif r == 3.0:
        return 1   # neutral
    elif r in [4.0, 5.0]:
        return 2   # positive
    return None

df["content"] = df["content"].apply(clean_text)
df["label"] = df["rating"].apply(map_label)

df = df.dropna(subset=["content", "label"])
df = df.drop_duplicates(subset=["content", "label"]).reset_index(drop=True)

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

df["label"] = df["label"].astype(int)
df["label"].value_counts()

,count
label,
2,255
0,25
1,16


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.4,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

dataset

DatasetDict({
    train: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 177
    })
    validation: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 59
    })
    test: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 60
    })
})

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_data(examples):
    encoding = tokenizer(
        examples["content"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    encoding["labels"] = examples["label"]
    return encoding

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
encoded_dataset = dataset.map(preprocess_data, batched=True)

cols_to_remove = [c for c in ["content", "rating", "label", "__index_level_0__"] if c in encoded_dataset["train"].column_names]
encoded_dataset = encoded_dataset.remove_columns(cols_to_remove)
encoded_dataset.set_format("torch")

encoded_dataset["train"][0]

Map:   0%|          | 0/177 [00:00<?, ? examples/s]

Map:   0%|          | 0/59 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

{'input_ids': tensor([ 101, 2573, 2092,  102,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    problem_type="single_label_classification",
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

df = pd.read_excel("shopify_reviews.xlsx")
df = df[["content", "rating"]].copy()
df.head()

,content,rating
0,i used to have a feliway - they can be interch...,5
1,NaN,5
2,NaN,5
3,"Excelente producto, los gatos se han calmado e...",5
4,NaN,5


In [ ]:
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = " ".join(x.split())
    if len(x) == 0:
        return None
    return x

def map_label(rating):
    try:
        r = float(rating)
    except:
        return None

    if r in [1.0, 2.0]:
        return 0   # negative
    elif r == 3.0:
        return 1   # neutral
    elif r in [4.0, 5.0]:
        return 2   # positive
    return None

df["content"] = df["content"].apply(clean_text)
df["label"] = df["rating"].apply(map_label)

df = df.dropna(subset=["content", "label"])
df = df.drop_duplicates(subset=["content", "label"]).reset_index(drop=True)

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

df["label"] = df["label"].astype(int)
df["label"].value_counts()

,count
label,
2,255
0,25
1,16


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.4,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

dataset

DatasetDict({
    train: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 177
    })
    validation: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 59
    })
    test: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 60
    })
})

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_data(examples):
    encoding = tokenizer(
        examples["content"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    encoding["labels"] = examples["label"]
    return encoding

In [ ]:
encoded_dataset = dataset.map(preprocess_data, batched=True)

cols_to_remove = [c for c in ["content", "rating", "label", "__index_level_0__"] if c in encoded_dataset["train"].column_names]
encoded_dataset = encoded_dataset.remove_columns(cols_to_remove)
encoded_dataset.set_format("torch")

encoded_dataset["train"][0]

Map:   0%|          | 0/177 [00:00<?, ? examples/s]

Map:   0%|          | 0/59 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

{'input_ids': tensor([ 101, 2573, 2092,  102,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    problem_type="single_label_classification",
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from deep_translator import GoogleTranslator
from tqdm import tqdm

tqdm.pandas()

def translate_text(text):
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except:
        return text

In [ ]:
df["content"] = df["content"].fillna("").astype(str)
df["content_en"] = df["content"].progress_apply(translate_text)

df[["content", "content_en"]].head()

100%|██████████| 296/296 [01:02<00:00,  4.70it/s]


,content,content_en
0,i used to have a feliway - they can be interch...,i used to have a feliway - they can be interch...
1,"Excelente producto, los gatos se han calmado e...","Excellent product, the cats have calmed down i..."
2,I noticed a difference in my girls within a co...,I noticed a difference in my girls within a co...
3,buen producto,good product
4,fits the diffuser no problem and i think its s...,fits the diffuser no problem and i think its s...


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

df = pd.read_excel("shopify_reviews.xlsx")
df = df[["content", "rating"]].copy()
df.head()

,content,rating
0,i used to have a feliway - they can be interch...,5
1,NaN,5
2,NaN,5
3,"Excelente producto, los gatos se han calmado e...",5
4,NaN,5


In [ ]:
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = " ".join(x.split())
    if len(x) == 0:
        return None
    return x

def map_label(rating):
    try:
        r = float(rating)
    except:
        return None

    if r in [1.0, 2.0]:
        return 0   # negative
    elif r == 3.0:
        return 1   # neutral
    elif r in [4.0, 5.0]:
        return 2   # positive
    return None

df["content"] = df["content"].apply(clean_text)
df["label"] = df["rating"].apply(map_label)

df = df.dropna(subset=["content", "label"])
df = df.drop_duplicates(subset=["content", "label"]).reset_index(drop=True)

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

df["label"] = df["label"].astype(int)
df["label"].value_counts()

,count
label,
2,255
0,25
1,16


In [ ]:
from deep_translator import GoogleTranslator
from tqdm import tqdm

tqdm.pandas()

def translate_text(text):
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except:
        return text  # hata olursa orijinali bırak

df["content_en"] = df["content"].progress_apply(translate_text)

df.head()

100%|██████████| 296/296 [00:17<00:00, 16.60it/s]


,content,rating,label,content_en
0,i used to have a feliway - they can be interch...,5,2,i used to have a feliway - they can be interch...
1,"Excelente producto, los gatos se han calmado e...",5,2,"Excellent product, the cats have calmed down i..."
2,I noticed a difference in my girls within a co...,5,2,I noticed a difference in my girls within a co...
3,buen producto,4,2,good product
4,fits the diffuser no problem and i think its s...,5,2,fits the diffuser no problem and i think its s...


## Load Shopify CSV dataset

Bu bölüm artık GoEmotions yerine kendi Shopify CSV dosyanı kullanır. Gerekli kolonlar: `content` ve `rating`.

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

df = pd.read_excel("shopify_reviews.xlsx")
df = df[["content", "rating"]].copy()
df.head()

,content,rating
0,i used to have a feliway - they can be interch...,5
1,NaN,5
2,NaN,5
3,"Excelente producto, los gatos se han calmado e...",5
4,NaN,5


In [ ]:
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = " ".join(x.split())
    if len(x) == 0:
        return None
    return x

def map_label(rating):
    try:
        r = float(rating)
    except:
        return None

    if r in [1.0, 2.0]:
        return 0   # negative
    elif r == 3.0:
        return 1   # neutral
    elif r in [4.0, 5.0]:
        return 2   # positive
    return None

df["content"] = df["content"].apply(clean_text)
df["label"] = df["rating"].apply(map_label)

df = df.dropna(subset=["content", "label"])
df = df.drop_duplicates(subset=["content", "label"]).reset_index(drop=True)

df.head()

,content,rating,label
0,i used to have a feliway - they can be interch...,5,2
1,"Excelente producto, los gatos se han calmado e...",5,2
2,I noticed a difference in my girls within a co...,5,2
3,buen producto,4,2
4,fits the diffuser no problem and i think its s...,5,2


In [ ]:
import os

file_name = 'shopify_reviews.xlsx'
if os.path.exists(file_name):
    print(f"The file '{file_name}' exists in the current directory.")
else:
    print(f"The file '{file_name}' does NOT exist in the current directory.")

The file 'shopify_reviews.xlsx' exists in the current directory.


## Prepare data

Bu bölümde metin temizleme, rating -> label dönüşümü ve train/validation/test split yapılır.

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = " ".join(x.split())
    if len(x) == 0:
        return None
    return x

def map_label(rating):
    try:
        r = float(rating)
    except:
        return None

    if r in [1.0, 2.0]:
        return 0   # negative
    elif r == 3.0:
        return 1   # neutral
    elif r in [4.0, 5.0]:
        return 2   # positive
    return None

df["content"] = df["content"].apply(clean_text)
df["label"] = df["rating"].apply(map_label)

df = df.dropna(subset=["content", "label"])
df = df.drop_duplicates(subset=["content", "label"]).reset_index(drop=True)

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

df["label"] = df["label"].astype(int)
df["label"].value_counts()

,count
label,
2,255
0,25
1,16


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.4,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

dataset

DatasetDict({
    train: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 177
    })
    validation: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 59
    })
    test: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 60
    })
})

## Tokenization

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_data(examples):
    encoding = tokenizer(
        examples["content"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    encoding["labels"] = examples["label"]
    return encoding

In [ ]:
encoded_dataset = dataset.map(preprocess_data, batched=True)

cols_to_remove = [c for c in ["content", "rating", "label", "__index_level_0__"] if c in encoded_dataset["train"].column_names]
encoded_dataset = encoded_dataset.remove_columns(cols_to_remove)
encoded_dataset.set_format("torch")

encoded_dataset["train"][0]

Map:   0%|          | 0/177 [00:00<?, ? examples/s]

Map:   0%|          | 0/59 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

{'input_ids': tensor([ 101, 2573, 2092,  102,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0

## Define model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    problem_type="single_label_classification",
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Train the model

In [ ]:
batch_size = 8
metric_name = "f1_macro"

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    "bert-shopify-sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    report_to="none",
    seed=42
)

## Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import EvalPrediction
import torch
import pandas as pd

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.argmax(preds, axis=1)
    labels = p.label_ids

    accuracy = accuracy_score(labels, preds)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }

## Start training

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,No log,0.480567,0.864407,0.288136,0.333333,0.309091,0.747199,0.864407,0.801541
2,No log,0.412606,0.864407,0.288136,0.333333,0.309091,0.747199,0.864407,0.801541
3,No log,0.422302,0.864407,0.288136,0.333333,0.309091,0.747199,0.864407,0.801541
4,No log,0.386686,0.864407,0.288136,0.333333,0.309091,0.747199,0.864407,0.801541


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=92, training_loss=0.5136852679045304, metrics={'train_runtime': 34.8062, 'train_samples_per_second': 20.341, 'train_steps_per_second': 2.643, 'total_flos': 23447147701248.0, 'train_loss': 0.5136852679045304, 'epoch': 4.0})

## Evaluate on validation and test

In [ ]:
trainer.evaluate()

{'eval_loss': 0.4805591106414795,
 'eval_accuracy': 0.864406779661017,
 'eval_precision_macro': 0.288135593220339,
 'eval_recall_macro': 0.3333333333333333,
 'eval_f1_macro': 0.3090909090909091,
 'eval_precision_weighted': 0.7471990807239299,
 'eval_recall_weighted': 0.864406779661017,
 'eval_f1_weighted': 0.8015408320493066,
 'eval_runtime': 0.2947,
 'eval_samples_per_second': 200.234,
 'eval_steps_per_second': 27.15,
 'epoch': 4.0}

In [ ]:
test_results = trainer.evaluate(encoded_dataset["test"])
test_results

{'eval_loss': 0.47895440459251404,
 'eval_accuracy': 0.8666666666666667,
 'eval_precision_macro': 0.2888888888888889,
 'eval_recall_macro': 0.3333333333333333,
 'eval_f1_macro': 0.30952380952380953,
 'eval_precision_weighted': 0.7511111111111112,
 'eval_recall_weighted': 0.8666666666666667,
 'eval_f1_weighted': 0.8047619047619048,
 'eval_runtime': 0.2172,
 'eval_samples_per_second': 276.211,
 'eval_steps_per_second': 36.828,
 'epoch': 4.0}

In [ ]:
pred_output = trainer.predict(encoded_dataset["test"])
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "neutral", "positive"],
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["negative", "neutral", "positive"],
    columns=["negative", "neutral", "positive"]
)
cm_df

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         5
     neutral       0.00      0.00      0.00         3
    positive       0.87      1.00      0.93        52

    accuracy                           0.87        60
   macro avg       0.29      0.33      0.31        60
weighted avg       0.75      0.87      0.80        60



,negative,neutral,positive
negative,0,0,5
neutral,0,0,3
positive,0,0,52


In [ ]:
import time
import psutil
import torch
import numpy as np
from sklearn.metrics import f1_score

In [ ]:
def measure_efficiency(trainer, tokenized_test):
    process = psutil.Process()

    ram_before = process.memory_info().rss / (1024 ** 2)

    gpu_peak = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()
    predictions = trainer.predict(tokenized_test)
    end_time = time.perf_counter()

    ram_after = process.memory_info().rss / (1024 ** 2)
    ram_used = ram_after - ram_before

    if torch.cuda.is_available():
        gpu_peak = torch.cuda.max_memory_allocated() / (1024 ** 2)

    y_true = predictions.label_ids
    y_pred = np.argmax(predictions.predictions, axis=1)

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    total_time = end_time - start_time
    time_per_sample = total_time / len(y_true)

    return {
        "total_inference_time_sec": total_time,
        "time_per_sample_sec": time_per_sample,
        "ram_before_mb": ram_before,
        "ram_after_mb": ram_after,
        "ram_used_mb": ram_used,
        "gpu_peak_mb": gpu_peak,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

In [ ]:
efficiency_results = measure_efficiency(trainer, encoded_dataset["test"])
efficiency_results

{'total_inference_time_sec': 0.2178668110000217,
 'time_per_sample_sec': 0.0036311135166670282,
 'ram_before_mb': 1557.1875,
 'ram_after_mb': 1557.1875,
 'ram_used_mb': 0.0,
 'gpu_peak_mb': 824.439453125,
 'macro_f1': 0.30952380952380953,
 'weighted_f1': 0.8047619047619048}

In [ ]:
predictions = trainer.predict(encoded_dataset["test"])
test_results = predictions.metrics
print(test_results)

{'test_loss': 0.47895440459251404, 'test_accuracy': 0.8666666666666667, 'test_precision_macro': 0.2888888888888889, 'test_recall_macro': 0.3333333333333333, 'test_f1_macro': 0.30952380952380953, 'test_precision_weighted': 0.7511111111111112, 'test_recall_weighted': 0.8666666666666667, 'test_f1_weighted': 0.8047619047619048, 'test_runtime': 0.3313, 'test_samples_per_second': 181.122, 'test_steps_per_second': 24.15}


In [ ]:
print(test_results)
print(test_results.keys())

{'test_loss': 0.47895440459251404, 'test_accuracy': 0.8666666666666667, 'test_precision_macro': 0.2888888888888889, 'test_recall_macro': 0.3333333333333333, 'test_f1_macro': 0.30952380952380953, 'test_precision_weighted': 0.7511111111111112, 'test_recall_weighted': 0.8666666666666667, 'test_f1_weighted': 0.8047619047619048, 'test_runtime': 0.3571, 'test_samples_per_second': 168.006, 'test_steps_per_second': 22.401}
dict_keys(['test_loss', 'test_accuracy', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro', 'test_precision_weighted', 'test_recall_weighted', 'test_f1_weighted', 'test_runtime', 'test_samples_per_second', 'test_steps_per_second'])


In [ ]:
import pandas as pd

final_results = {
    "model": model.name_or_path,
    "accuracy": test_results["test_accuracy"],
    "precision_macro": test_results["test_precision_macro"],
    "recall_macro": test_results["test_recall_macro"],
    "f1_macro": test_results["test_f1_macro"],
    "precision_weighted": test_results["test_precision_weighted"],
    "recall_weighted": test_results["test_recall_weighted"],
    "f1_weighted": test_results["test_f1_weighted"],
    "runtime_sec": test_results["test_runtime"],
    "samples_per_second": test_results["test_samples_per_second"],
    "steps_per_second": test_results["test_steps_per_second"],
    "ram_used_mb": efficiency_results["ram_used_mb"],
    "gpu_peak_mb": efficiency_results["gpu_peak_mb"]
}

results_df = pd.DataFrame([final_results])
results_df

,model,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,runtime_sec,samples_per_second,steps_per_second,ram_used_mb,gpu_peak_mb
0,distilbert-base-uncased,0.866667,0.288889,0.333333,0.309524,0.751111,0.866667,0.804762,0.3571,168.006,22.401,0.0,824.439453
